# 단층 퍼셉트론
단층 퍼셉트론(Sin이e Layer Perceptron)은 하나의 계층을 갖는 모델을 의미한다. 입력을 통해 데이
터가 전달되고 입력값(z)은 각각의 가중치와 함께 노드에 전달된다. 전달된 입력값(아과 가중치를 곱한
값이 활성화 함수에 전달된다.
활성화 함수에서 출력값아)이 계산되고 이 값을 손실 함수에 실젯값(짜과 함께 연산해 가중치를 변경한
다. 단층 퍼셉트론은 앞선 강좌에서 다룬 모델 구조와 동일한 형태가 된다. 그림 3.27은 지금까지의 모
델 구조를 시각화한 것이다.  
![nn](.\image\perceptronimage.png)


In [1]:
import torch
import pandas as pd
from torch import nn
from torch import optim
from torch.utils.data import Dataset, DataLoader


class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x1 = df.iloc[:, 0].values
        self.x2 = df.iloc[:, 1].values
        self.y = df.iloc[:, 2].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x1[index], self.x2[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y

    def __len__(self):
        return self.length

class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer = nn.Sequential(
            nn.Linear(2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.layer(x)
        return x

In [2]:
train_dataset = CustomDataset("../datasets/perceptron.csv")
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CustomModel().to(device)
criterion = nn.BCELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 퍼셉트론 모델 실습
Xl, x2는 입력값을 의미하며, y는 XOR 게이트를 통과했을 때의 결과를 의미한다. 이 데이터를 활용해
단층 퍼셉트론과 다층 퍼셉트론 구조의 인공 신경망 모델을 구현해 본다.23 다음 예제 3.58은 단층 퍼셉
트론 구조를 보여준다.


In [4]:
for epoch in range(10000):
    cost = 0.0

    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = criterion(output, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(train_dataloader)

    if (epoch + 1) % 1000 == 0:
        print(f"Epoch : {epoch+1:4d}, Cost : {cost:.3f}")

C:\Users\H11\AppData\Local\Temp\ipykernel_24136\999614697.py:17: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  x = torch.FloatTensor([self.x1[index], self.x2[index]])
C:\Users\H11\AppData\Local\Temp\ipykernel_24136\999614697.py:18: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  y = torch.FloatTensor([self.y[index]])


Epoch : 1000, Cost : 0.692
Epoch : 2000, Cost : 0.692
Epoch : 3000, Cost : 0.692
Epoch : 4000, Cost : 0.692
Epoch : 5000, Cost : 0.692
Epoch : 6000, Cost : 0.692
Epoch : 7000, Cost : 0.692
Epoch : 8000, Cost : 0.692
Epoch : 9000, Cost : 0.692
Epoch : 10000, Cost : 0.692


# 단층 퍼셉트론의 한계
단층 퍼셉트론은 AND, OR, NAND 게이트와 같은 구조를 갖는 모델은 쉽게 구현할 수 있다. 예를 들
어 OR 게이트를 그래프로 표현한다면 그림 3.28과 같이 그릴 수 있다.

![nn](./image/limitsingleperceptron.png)
하나의 기울기로 값을 나누고, 그 기울기보다 위에 있거나 아래에 있다면 참 또는 거짓 값을 갖게 할 수
있다. 하지만 XOR 게이트처럼 하나의 기울기로 표현하기 어려운 구조에서는 단층 퍼셉트론을 적용하기
가 어렵다.

그림 3.28의 데이터에서 XOR을 표현하려면 [(0, 0)] / [(0, 1), (1, 0)] / [(1, 1)]의 구조로 삼등분해야
한다. [(0, 0), (1, 1)] / [(0, 1), (1, 0)]의 구조로 이등분하려 한다면 직선이 아닌 곡선의 형태가 되어
학습이 어려워질 뿐만 아니라 과대적합 문제도 발생한다.

In [5]:
with torch.no_grad():
    model.eval()
    inputs = torch.FloatTensor([
        [0, 0],
        [0, 1],
        [1, 0],
        [1, 1]
    ]).to(device)
    outputs = model(inputs)
    
    print("---------")
    print(outputs)
    print(outputs <= 0.5)

---------
tensor([[0.4669],
        [0.4990],
        [0.5021],
        [0.5341]], device='cuda:0')
tensor([[ True],
        [ True],
        [False],
        [False]], device='cuda:0')
